# PLDDT Analysis for Priority Unstudied Genes (PUGs)

This notebook fetches PLDDT (Predicted Local Distance Difference Test) metrics from the AlphaFold API for each UniProt ID in the priority unstudied genes dataset.

## Workflow:
1. Load priority unstudied genes data
2. Extract UniProt IDs 
3. Query AlphaFold API for PLDDT metrics
4. Save results to intermediate files directory

## Output Metrics:
- `modelEntityId`: UniProt identifier used in AlphaFold
- `globalMetricValue`: Overall confidence score
- `fractionPlddtVeryLow`: Fraction with very low confidence (0-50)
- `fractionPlddtLow`: Fraction with low confidence (50-70)
- `fractionPlddtConfident`: Fraction with confident predictions (70-90)
- `fractionPlddtVeryHigh`: Fraction with very high confidence (90-100)

Prepared using Claude Sonnet 4

In [33]:
# Import required libraries
import pandas as pd
import requests
import time
import json
import os
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

print("Libraries imported successfully!")

Libraries imported successfully!


In [34]:
# Define file paths
DATA_PATH = r"G:\My Drive\Crick_LMS\projects\a_unstudied_genes\priority_unstudied_genes.tsv"
INTERMEDIATE_DIR = r"G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files"
OUTPUT_FILE = os.path.join(INTERMEDIATE_DIR, "plddt_for_PUGs.tsv")

# Create intermediate directory if it doesn't exist
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_FILE}")
print(f"Intermediate directory created/verified: {INTERMEDIATE_DIR}")

Data path: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\priority_unstudied_genes.tsv
Output path: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files\plddt_for_PUGs.tsv
Intermediate directory created/verified: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files


In [35]:
# Load the priority unstudied genes data
print("Loading priority unstudied genes data...")
pugs_df = pd.read_csv(DATA_PATH, sep='\t')

print(f"Loaded {len(pugs_df)} genes from priority unstudied genes dataset")
print(f"Columns: {list(pugs_df.columns)}")
print("\nFirst few rows:")
print(pugs_df.head())

# Extract UniProt IDs
uniprot_ids = pugs_df['UniProt ID'].dropna().unique()
print(f"\nFound {len(uniprot_ids)} unique UniProt IDs")
print("Sample UniProt IDs:", uniprot_ids[:5])

Loading priority unstudied genes data...
Loaded 122 genes from priority unstudied genes dataset
Columns: ['Systematic ID', 'Gene name', 'Product description', 'UniProt ID', 'Synonyms', 'Feature type', 'Start position', 'End position', 'Chromosome', 'Strand', 'Budding yeast orthologs', 'S. japonicus orthologs', 'Human orthologs', 'Physical interactors', 'Deletion viability', 'Transmembrane domain', 'Characterisation status', 'Taxonomic distribution', 'PDB structure IDs', 'Paralog IDs', 'RNACentral ID', 'Protein molecular weight', 'Protein length', 'Protein length (bins)', 'Spliced RNA length', 'Unspliced RNA length']

First few rows:
  Systematic ID Gene name                                Product description  \
0   SPBC1711.12     ppp16  acylpeptide hydrolase, implicated in protein q...   
1   SPCP1E11.10      dhm1    ankyrin repeat protein, unknown biological role   
2  SPBC20F10.03       NaN  armadillo-type fold protein, human IFRD1 ortho...   
3  SPCC16C4.02c      asi1  armadillo-ty

In [36]:
# Define function to fetch PLDDT data from AlphaFold API
def fetch_alphafold_plddt(uniprot_id, max_retries=3, delay=0.1):
    """
    Fetch PLDDT metrics for a given UniProt ID from AlphaFold API
    
    Parameters:
    uniprot_id (str): UniProt identifier
    max_retries (int): Maximum number of retry attempts
    delay (float): Delay between requests in seconds (reduced for parallelization)
    
    Returns:
    dict: Dictionary containing PLDDT metrics or None if failed
    """
    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200:
                data = response.json()
                
                # Extract summary metrics if available
                if data and len(data) > 0:
                    entry = data[0]  # Take first entry.  There is not usually more than one.
                    
                    # Extract PLDDT summary metrics
                    #summary = entry.get('summaryMetrics', {})
                    #plddt_summary = summary.get('plddtMetrics', {})
                    
                    result = {
                        'modelEntityId': entry.get('entryId', uniprot_id),
                        'globalMetricValue': entry.get('globalMetricValue', None),
                        'fractionPlddtVeryLow': entry.get('fractionPlddtVeryLow', None),
                        'fractionPlddtLow': entry.get('fractionPlddtLow', None),
                        'fractionPlddtConfident': entry.get('fractionPlddtConfident', None),
                        'fractionPlddtVeryHigh': entry.get('fractionPlddtVeryHigh', None)
                    }
                    
                    # Reduced delay for parallel execution
                    if delay > 0:
                        time.sleep(delay)
                    return result
                    
            elif response.status_code == 404:
                # No AlphaFold structure available for this UniProt ID
                return {
                    'modelEntityId': uniprot_id,
                    'globalMetricValue': None,
                    'fractionPlddtVeryLow': None,
                    'fractionPlddtLow': None,
                    'fractionPlddtConfident': None,
                    'fractionPlddtVeryHigh': None,
                    'error': 'No AlphaFold structure available'
                }
            elif response.status_code == 429:
                # Rate limited - wait longer
                print(f"Rate limited for {uniprot_id}, waiting...")
                time.sleep(delay * (attempt + 1) * 5)
            else:
                print(f"Attempt {attempt + 1}: HTTP {response.status_code} for {uniprot_id}")
                time.sleep(delay * (attempt + 1))  # Exponential backoff
                
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1}: Network error for {uniprot_id}: {e}")
            time.sleep(delay * (attempt + 1))
    
    # If all attempts failed
    return {
        'modelEntityId': uniprot_id,
        'globalMetricValue': None,
        'fractionPlddtVeryLow': None,
        'fractionPlddtLow': None,
        'fractionPlddtConfident': None,
        'fractionPlddtVeryHigh': None,
        'error': 'API request failed after retries'
    }

print("AlphaFold API function defined successfully!")

AlphaFold API function defined successfully!


In [37]:
# Fetch PLDDT data for all UniProt IDs using parallel processing
print(f"Fetching PLDDT data for {len(uniprot_ids)} UniProt IDs using parallel processing...")
print("This should be much faster than sequential processing!")

# Configuration for parallel processing
MAX_WORKERS = 10  # Number of concurrent API requests (adjust if needed)
BATCH_SIZE = 50   # Process in batches to avoid overwhelming the API

plddt_results = []

# Process in batches to be respectful to the API
for batch_start in range(0, len(uniprot_ids), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(uniprot_ids))
    batch_ids = uniprot_ids[batch_start:batch_end]
    
    print(f"\nProcessing batch {batch_start//BATCH_SIZE + 1}/{(len(uniprot_ids)-1)//BATCH_SIZE + 1} "
          f"(UniProt IDs {batch_start+1}-{batch_end})")
    
    batch_results = []
    
    # Use ThreadPoolExecutor for concurrent API requests
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all requests in the batch
        future_to_uniprot = {
            executor.submit(fetch_alphafold_plddt, uniprot_id): uniprot_id 
            for uniprot_id in batch_ids
        }
        
        # Collect results with progress bar
        with tqdm(total=len(batch_ids), desc=f"Batch {batch_start//BATCH_SIZE + 1}") as pbar:
            for future in as_completed(future_to_uniprot):
                uniprot_id = future_to_uniprot[future]
                try:
                    result = future.result()
                    batch_results.append(result)
                except Exception as exc:
                    print(f"Error processing {uniprot_id}: {exc}")
                    # Add error result
                    batch_results.append({
                        'modelEntityId': uniprot_id,
                        'globalMetricValue': None,
                        'fractionPlddtVeryLow': None,
                        'fractionPlddtLow': None,
                        'fractionPlddtConfident': None,
                        'fractionPlddtVeryHigh': None,
                        'error': f'Processing error: {exc}'
                    })
                pbar.update(1)
    
    # Add batch results to main results
    plddt_results.extend(batch_results)
    
    # Brief pause between batches to be respectful
    if batch_end < len(uniprot_ids):
        print(f"Completed batch. Brief pause before next batch...")
        time.sleep(2)

print(f"\n✅ Completed fetching data for {len(plddt_results)} entries")

# Convert results to DataFrame
plddt_df = pd.DataFrame(plddt_results)

print(f"Created DataFrame with shape: {plddt_df.shape}")
print(f"Columns: {list(plddt_df.columns)}")
print("\nSample results:")
print(plddt_df.head())

Fetching PLDDT data for 122 UniProt IDs using parallel processing...
This should be much faster than sequential processing!

Processing batch 1/3 (UniProt IDs 1-50)


Batch 1: 100%|██████████| 50/50 [00:27<00:00,  1.82it/s]



Completed batch. Brief pause before next batch...

Processing batch 2/3 (UniProt IDs 51-100)

Processing batch 2/3 (UniProt IDs 51-100)


Batch 2: 100%|██████████| 50/50 [00:32<00:00,  1.56it/s]



Completed batch. Brief pause before next batch...

Processing batch 3/3 (UniProt IDs 101-122)

Processing batch 3/3 (UniProt IDs 101-122)


Batch 3: 100%|██████████| 22/22 [00:19<00:00,  1.13it/s]


✅ Completed fetching data for 122 entries
Created DataFrame with shape: (122, 6)
Columns: ['modelEntityId', 'globalMetricValue', 'fractionPlddtVeryLow', 'fractionPlddtLow', 'fractionPlddtConfident', 'fractionPlddtVeryHigh']

Sample results:
  modelEntityId  globalMetricValue  fractionPlddtVeryLow  fractionPlddtLow  \
0  AF-Q9UU77-F1              77.62                 0.164             0.179   
1  AF-Q9P7S8-F1              77.25                 0.137             0.178   
2  AF-O94536-F1              79.06                 0.109             0.096   
3  AF-O74447-F1              83.31                 0.113             0.040   
4  AF-O42973-F1              81.50                 0.105             0.110   

   fractionPlddtConfident  fractionPlddtVeryHigh  
0                   0.203                  0.454  
1                   0.300                  0.385  
2                   0.460                  0.335  
3                   0.248                  0.599  
4                   0.318         

In [38]:
plddt_df 

,modelEntityId,globalMetricValue,fractionPlddtVeryLow,fractionPlddtLow,fractionPlddtConfident,fractionPlddtVeryHigh
0,AF-Q9UU77-F1,77.62,0.164,0.179,0.203,0.454
1,AF-Q9P7S8-F1,77.25,0.137,0.178,0.300,0.385
2,AF-O94536-F1,79.06,0.109,0.096,0.460,0.335
3,AF-O74447-F1,83.31,0.113,0.040,0.248,0.599
4,AF-O42973-F1,81.50,0.105,0.110,0.318,0.466
...,...,...,...,...,...,...
117,AF-O42948-F1,70.88,0.144,0.237,0.588,0.031
118,AF-Q9P374-F1,84.62,0.085,0.056,0.288,0.571
119,AF-Q9URW3-F1,83.94,0.168,0.084,0.031,0.718
120,AF-Q9HFF2-F1,72.88,0.119,0.312,0.468,0.101


In [39]:
# Analyze and summarize results
print("=== PLDDT Data Analysis Summary ===")

# Basic statistics
print(f"Total entries processed: {len(plddt_df)}")
print(f"Entries with valid PLDDT data: {plddt_df['globalMetricValue'].notna().sum()}")
print(f"Entries without AlphaFold structures: {plddt_df['globalMetricValue'].isna().sum()}")

# Check for any errors
if 'error' in plddt_df.columns:
    error_counts = plddt_df['error'].value_counts()
    print(f"\nError summary:")
    print(error_counts)

# Summary statistics for valid entries
valid_data = plddt_df[plddt_df['globalMetricValue'].notna()]
if len(valid_data) > 0:
    print(f"\n=== Valid PLDDT Statistics ===")
    print(f"Global metric value range: {valid_data['globalMetricValue'].min():.2f} - {valid_data['globalMetricValue'].max():.2f}")
    print(f"Mean global metric value: {valid_data['globalMetricValue'].mean():.2f}")
    
    # Confidence distribution summary
    print(f"\n=== Confidence Distribution (mean values) ===")
    print(f"Very Low confidence (0-50): {valid_data['fractionPlddtVeryLow'].mean():.3f}")
    print(f"Low confidence (50-70): {valid_data['fractionPlddtLow'].mean():.3f}")  
    print(f"Confident (70-90): {valid_data['fractionPlddtConfident'].mean():.3f}")
    print(f"Very High confidence (90-100): {valid_data['fractionPlddtVeryHigh'].mean():.3f}")

print("\nData preview:")
print(plddt_df.head(10))

=== PLDDT Data Analysis Summary ===
Total entries processed: 122
Entries with valid PLDDT data: 122
Entries without AlphaFold structures: 0

=== Valid PLDDT Statistics ===
Global metric value range: 34.56 - 97.69
Mean global metric value: 81.94

=== Confidence Distribution (mean values) ===
Very Low confidence (0-50): 0.121
Low confidence (50-70): 0.117
Confident (70-90): 0.210
Very High confidence (90-100): 0.553

Data preview:
  modelEntityId  globalMetricValue  fractionPlddtVeryLow  fractionPlddtLow  \
0  AF-Q9UU77-F1              77.62                 0.164             0.179   
1  AF-Q9P7S8-F1              77.25                 0.137             0.178   
2  AF-O94536-F1              79.06                 0.109             0.096   
3  AF-O74447-F1              83.31                 0.113             0.040   
4  AF-O42973-F1              81.50                 0.105             0.110   
5  AF-Q9HDX5-F1              82.56                 0.038             0.128   
6  AF-Q9P778-F1      

In [40]:
# Save results to TSV file
print("Saving results to TSV file...")

# Filter to only the required columns and remove any error columns for final output
output_columns = ['modelEntityId', 'globalMetricValue', 'fractionPlddtVeryLow', 
                 'fractionPlddtLow', 'fractionPlddtConfident', 'fractionPlddtVeryHigh']

# Create output dataframe with only required columns
output_df = plddt_df[output_columns].copy()

# Save to TSV file
output_df.to_csv(OUTPUT_FILE, sep='\t', index=False, na_rep='NA')

print(f"Successfully saved {len(output_df)} entries to: {OUTPUT_FILE}")

# Display final summary
print(f"\n=== Final Output Summary ===")
print(f"File: {OUTPUT_FILE}")
print(f"Shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")

# Show sample of saved data
print(f"\nSample of saved data:")
print(output_df.head(10).to_string())

print(f"\n✅ PLDDT analysis completed successfully!")
print(f"Results saved to: {OUTPUT_FILE}")

Saving results to TSV file...
Successfully saved 122 entries to: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files\plddt_for_PUGs.tsv

=== Final Output Summary ===
File: G:\My Drive\Crick_LMS\projects\a_unstudied_genes\intermediate_files\plddt_for_PUGs.tsv
Shape: (122, 6)
Columns: ['modelEntityId', 'globalMetricValue', 'fractionPlddtVeryLow', 'fractionPlddtLow', 'fractionPlddtConfident', 'fractionPlddtVeryHigh']

Sample of saved data:
  modelEntityId  globalMetricValue  fractionPlddtVeryLow  fractionPlddtLow  fractionPlddtConfident  fractionPlddtVeryHigh
0  AF-Q9UU77-F1              77.62                 0.164             0.179                   0.203                  0.454
1  AF-Q9P7S8-F1              77.25                 0.137             0.178                   0.300                  0.385
2  AF-O94536-F1              79.06                 0.109             0.096                   0.460                  0.335
3  AF-O74447-F1              83.31                 0.11